# Water quality modeling with WNTR

How do water age, water sources, and chlorine residuals vary across a distribution system?

Today: **(1) water age, (2) source tracing, (3) chlorine decay.**

We will use **Net3** for all three analyses.

## **1. Load the network**

In [ ]:
try:
    import wntr
except ImportError:
    %pip install wntr
    import wntr
import matplotlib.pyplot as plt

In [ ]:
# Update the path if your Net3 file is in another folder
inp = 'Net3.inp'
wn = wntr.network.WaterNetworkModel(inp)

print('Reservoirs:', wn.reservoir_name_list)
print('Tanks:', wn.tank_name_list)
print('Simulation duration (hours):', wn.options.time.duration / 3600)

wntr.graphics.plot_network(wn, title='Net3 network', node_size=20, link_width=1)
plt.show()

## **2. Water age: where is the oldest water?**
Water age measures the time since water entered the system. EPANET reports water age in **seconds**; we convert it to hours.

### Setting up a water age simulation

Before running the simulation, we need to define a few settings:

- **Water quality parameter (`AGE`):** Tells EPANET to track how long water has been in the distribution system. Water enters from reservoirs with an age of zero.
- **Simulation duration (7 days):** Water age takes time to stabilize, especially in tanks and areas with low flow. We therefore need to simulate several days.
- **Water quality time step (5 minutes):** How often EPANET calculates changes in water quality as water moves through the network.
- **Reporting time step (1 hour):** How often simulation results are saved. This can be longer than the calculation time step to reduce the amount of output data.

The hydraulic model is solved first, and the resulting flows are used to simulate water age throughout the network.

**Think about it:** Why do we need to simulate 7 days if the demand pattern repeats every 24 hours?

In [ ]:
# Start with a fresh copy of the network
wn = wntr.network.WaterNetworkModel(💡)

# define simulation settings
wn.options.time.💡 = 💡     # set simulation duration to 7 days
wn.options.time.💡 = 💡     # set quality time step to 5 minutes
wn.options.time.💡 = 💡     # report results every hour

# define water quality model
wn.options.quality.💡 = 💡

# run simulation and get results
sim = wntr.sim.EpanetSimulator(wn)
results_age = sim.run_sim()

# get age at all nodes in hours
age = 💡.💡
age.head()

### **A. Compare four junctions**
Plot the water age over the 7-day simulation for four junctions.


In [ ]:
# select locations to plot

nodes_to_plot = ['105', '187', '231', '275']

time_hours = age.index / 3600   # convert time to hours

# plot age at the selected nodes

plt.plot(💡, 💡)
plt.xlabel('Time (hours)')
plt.ylabel('Water age (hours)')
plt.title('Water age at four junctions')
plt.legend(nodes_to_plot)
plt.show()

**Questions:**
1. How does water age differ among the four junctions?
2. What explains the daily fluctuations in water age?
3. Has water age reached a repeating daily pattern at all four junctions? How could you check?
4. Why does the water age at some junctions continue to increase over time? What might this suggest about water circulation or storage in the system? What would you examine in the model to investigate?

### **B. Where is the oldest water?**

In [ ]:
# Plot water age at all junctions at the end of the simulation
final_age = age.iloc[💡][wn.junction_name_list]

wntr.graphics.plot_network(wn, node_attribute = 💡,
                           title='Water age at the end of day 7 (hours)',
                           node_size=20, link_width=0.5,
                           node_cmap='coolwarm', add_colorbar=True)
plt.show()

print('Five junctions with highest water age (hours):')
print(final_age.sort_values(ascending=False).head())

**Questions:**
1. Are the oldest nodes necessarily the farthest from a reservoir?
2. What else might explain their age?

## **3. Source tracing: where does the water come from?**
A tracer reports the **percentage** of water at each node that originated at the selected source.

In a distribution system with multiple water sources and storage tanks, how can we determine where the water supplied to customers comes from and how these contributions change over time?

**Task:** Use source tracing to determine when water from the Lake and River reaches different parts of the network and what percentage of the water comes from each source.

In [ ]:
# set up the model
wn = wntr.network.WaterNetworkModel(inp)
wn.options.time.duration = 3 * 24 * 3600
wn.options.time.quality_timestep = 5 * 60
wn.options.time.report_timestep = 3600

# Run the Lake tracer
wn.options.💡 = 💡
wn.options.💡

# run simulation
sim = wntr.sim.EpanetSimulator(wn)
results_lake = sim.run_sim()

# get results
lake_trace = results_lake.node[💡]  # percent from Lake

# Run the River tracer using the same network


### **A. One junction, two sources**
Select one location and show what percentage of the water comes from the Lake

In [ ]:

node = 💡     # try another junction after inspecting the results
time_hours = lake_trace.index / 3600

plt.plot(time_hours, lake_trace[node], label='Lake')
plt.xlabel('Time (hours)')
plt.ylabel('Source contribution (%)')
plt.title('Source tracing at junction ' + node)
plt.legend()
plt.show()

💡Repeat and plot what percentage of the water comes from each source (Lake and River).

**Questions:**
1. When does the dominant source change?
2. Do the Lake and River contributions always add to 100%? If not, what other water might be present?

### **B. Compare source influence across the network**

In [ ]:
# Look at the same hour on the final simulated day
hour = 60  # hours since the start
lake_at_hour = lake_trace.loc[hour * 3600]
river_at_hour = river_trace.loc[hour * 3600]

wntr.graphics.plot_network(wn, node_attribute=lake_at_hour,
                           title='Lake contribution at hour ' + str(hour),
                           node_size=20, link_width=0.5,
                           node_cmap='coolwarm', node_range=[0, 100],
                           add_colorbar=True)
plt.show()

wntr.graphics.plot_network(wn, node_attribute=river_at_hour,
                           title='River contribution at hour ' + str(hour),
                           node_size=20, link_width=0.5,
                           node_cmap='coolwarm', node_range=[0, 100],
                           add_colorbar=True)
plt.show()

**Try it:** Change `hour` to 48 or 72. Which areas change source contribution? How could a utility use this information?

In [ ]:
# Compare source contributions at four times on the final day
hours = [6, 12, 18, 24]
last_day = wn.options.time.duration / 3600 - 24

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for i, hour in enumerate(hours):
    t = (last_day + hour) * 3600

    # Lake contribution
    wntr.graphics.plot_network(
        wn, node_attribute=lake_trace.loc[t],
        ax=axes[0, i],
        title=f'Lake: Hour {hour}',
        node_size=15, link_width=0.5,
        node_cmap='coolwarm', node_range=[0, 100],
        add_colorbar=False, show_plot=False
    )

    # River contribution
    wntr.graphics.plot_network(
        wn, node_attribute=river_trace.loc[t],
        ax=axes[1, i],
        title=f'River: Hour {hour}',
        node_size=15, link_width=0.5,
        node_cmap='coolwarm', node_range=[0, 100],
        add_colorbar=False, show_plot=False
    )

plt.tight_layout()
plt.show()

## **4. Chlorine: how does the residual change?**
Chlorine is transported with the water and decays as it travels. We will use **first-order bulk decay**, with no wall reaction, to keep the example simple.

**Predict:** Which of the four junctions should have the lowest chlorine residual? Compare with the water age plot.

In [ ]:
# Set up a fresh chlorine model
wn = wntr.network.WaterNetworkModel(inp)
wn.options.time.duration = 7 * 24 * 3600
wn.options.time.quality_timestep = 5 * 60
wn.options.time.report_timestep = 3600

# set up water quality simulation
wn.options.quality.parameter = 💡
wn.options.quality.chemical_name = 💡

# WNTR stores chemical concentration internally in kg/m3.
# 1 mg/L = 0.001 kg/m3
MG_L_TO_KG_M3 = 0.001

# Reaction coefficients are stored in 1/s; we specify the rate in 1/day
wn.options.reaction.💡 = 💡
wn.options.reaction.💡 = 💡
wn.options.reaction.💡 = 💡

# define 3 mg/L at both reservoirs

source = wn.get_node('Lake')
source.initial_quality = 3.0 * MG_L_TO_KG_M3

source = wn.get_node('River')
source.initial_quality = 3.0 * MG_L_TO_KG_M3

# simulate and get results
sim = wntr.sim.EpanetSimulator(wn)
results_chlorine = sim.run_sim()

chlorine = results_chlorine.node['quality'] / 💡  # convert to mg/L

### **A. Chlorine at the same four junctions**

In [ ]:
time_hours = chlorine.index / 3600

plt.plot(time_hours, chlorine[💡])

plt.xlabel('Time (hours)')
plt.ylabel('Chlorine residual (mg/L)')
plt.title('Chlorine at four junctions')
plt.legend(nodes_to_plot)
plt.show()

**Questions:**

1. How does chlorine residual vary among the four junctions? Which junction has the lowest residual, and why?
2. Why does chlorine residual fluctuate throughout the day, even though the chlorine concentration at the sources is constant?
3. Compare these results with the water age simulation. What relationship do you observe between water age and chlorine residual?
4. Why does chlorine take longer to reach junction 231? What does this tell us about water transport to this location?

### **B. Map the residual**

In [ ]:
final_chlorine = chlorine.iloc[💡][wn.junction_name_list]

wntr.graphics.plot_network(wn, node_attribute = 💡,
                           title='Chlorine residual at the end of day 7 (mg/L)',
                           node_size=30, link_width=0.5,
                           node_cmap='coolwarm_r', add_colorbar=True)
plt.show()

### **C. What happens if chlorine decays faster?**
Suppose the water temperature increases. Compare a slower and faster bulk decay rate.

In [ ]:
# Change only the bulk decay coefficient, then run again
wn.options.reaction.bulk_coeff = 💡 / (24 * 3600)  # 1/day

sim = wntr.sim.EpanetSimulator(wn)
results_fast = sim.run_sim()

chlorine_fast = results_fast.node['quality'] / MG_L_TO_KG_M3

# Compare one junction under both decay rates
node = '187'
plt.plot(time_hours, chlorine[💡], label='k = -0.3 /day')
plt.plot(time_hours, chlorine_fast[💡], label='k = -1.0 /day')
plt.xlabel('Time (hours)')
plt.ylabel('Chlorine residual (mg/L)')
plt.title('Effect of chlorine decay at junction ' + node)
plt.legend()
plt.show()

**Questions:**
1. Which areas will show the largest change?
2. Would increasing the chlorine concentration at the treatment plant solve low residuals everywhere? What might be the tradeoff?


## **5. Tank mixing: does the mixing model matter?**

EPANET offers different assumptions about how water mixes inside tanks. We will compare **complete mixing (MIXED)**, **first-in, first-out (FIFO)**, and **last-in, first-out (LIFO)** using **water age**. The tank and all other model settings stay the same.

**Predict:** Which model will produce the oldest water leaving the tank? Will the differences be constant over time?


In [ ]:
# Set up a fresh water quality model
wn = wntr.network.WaterNetworkModel(inp)
print('Available tanks:', wn.tank_name_list)

# set up simulation
wn = wntr.network.WaterNetworkModel(inp)
wn.options.time.duration = 7 * 24 * 3600
wn.options.time.quality_timestep = 5 * 60
wn.options.time.report_timestep = 3600

# water age model
wn.options.quality.parameter = 💡

# Compare three mixing assumptions for one tank
tank_name = wn.tank_name_list[💡]      # select tank
print('Selected tank:', tank_name)


tank = wn.get_node(tank_name)         # get tank

# mixing models 'MIXED', 'FIFO', 'LIFO'

# MIXED model
mixing = 💡
tank.💡= 💡

sim = wntr.sim.EpanetSimulator(wn)
results = sim.run_sim()
tank_age_mixed = results.node['quality'][tank_name] / 3600  # hours



# Plot water age in the tank
plt.figure()
plt.plot(tank_age.index / 3600, tank_age_mixed, label = 'mixed')


plt.xlabel('Time (hours)')
plt.ylabel('Water age (hours)')
plt.title('Effect of tank mixing model: Tank ' + tank_name)
plt.legend()
plt.show()



💡Repeat for FIFO and LIFO and show on the same plot



**Questions:**
1. How do the three water-age curves differ? Why might FIFO and LIFO produce different results?
2. Change `tank_name` to another ID from `wn.tank_name_list` and compare again. Do all tanks respond the same way?
